# Recommendation Agent - UI-Optimized Experimental Notebook

This notebook deconstructs the `RecommendationAgent` with **UI-specific optimizations** for concise, actionable output.

## Key Features:
- ✅ **Story Alignment**: Explains WHY business rules were selected and their relevance
- ✅ **Specificity Enforcement**: Concrete identifiers (CPT codes, policy IDs, amounts)
- ✅ **UI Brevity**: Word count limits for space-constrained displays
- ✅ **Validation with Retry**: Automatic retry with feedback (max 2 attempts)
- ✅ **Quality Metrics**: Dashboard showing compliance

## Word Count Limits:
- Description: MAX 100 words
- Evidence: MAX 15 words per bullet
- Story Alignment: MAX 30 words per bullet
- Peer Benchmarking: MAX 25 words per bullet

## Workflow:
1. Setup environment and configuration
2. Prepare input data
3. Load decision tree rules
4. Build UI-optimized prompts
5. Invoke LLM with retry logic
6. Validate output (story alignment, specificity, brevity)
7. Display quality metrics
8. Export results

## 1. Setup & Imports

In [1]:
import sys
import json
import re
import yaml
from pathlib import Path
from typing import Any, Dict, List, Optional
from datetime import datetime

# Add packages to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / "packages" / "agents" / "src"))
sys.path.insert(0, str(project_root / "packages" / "core" / "src"))
sys.path.insert(0, str(project_root / "packages" / "utils" / "src"))

print(f"Project root: {project_root}")
print("✓ Imports successful")

Project root: c:\projects\coc\idiscovery-deep-research
✓ Imports successful


In [2]:
from deep_research_agents.decision_tree_rules import DecisionTreeRuleEngine

print("✓ Agent modules imported")

2026-05-27 17:17:14,914 - policy_extractor.system - INFO - === Policy Extractor Logging Initialized ===
2026-05-27 17:17:14,916 - policy_extractor.system - INFO - Log directory: c:\projects\coc\idiscovery-deep-research\notebooks\prototyping\logs
2026-05-27 17:17:14,916 - policy_extractor.system - INFO - Max file size: 50.0MB
2026-05-27 17:17:14,917 - policy_extractor.system - INFO - Backup count: 10
2026-05-27 17:17:14,918 - policy_extractor.system - INFO - Console output enabled: True
2026-05-27 17:17:14,918 - policy_extractor.system - INFO - Console log level: INFO
2026-05-27 17:17:14,919 - policy_extractor.system - INFO - Console stream: stdout
2026-05-27 17:17:14,919 - policy_extractor.system - INFO - Process ID: 9832
2026-05-27 17:17:14,920 - policy_extractor.system - INFO - Component log levels:
2026-05-27 17:17:14,921 - policy_extractor.system - INFO -   policy_extractor.snowflake_store: WARNING
2026-05-27 17:17:14,922 - policy_extractor.system - INFO -   policy_extractor.snowfl

## 2. Configuration (Editable)

In [3]:
# ============= UI-OPTIMIZED CONFIGURATION =============

# Decision tree rules
USE_DECISION_TREE = True  # Set to False to disable
DECISION_TREE_PATH = project_root / "configs" / "decision_tree_rules.yaml"

# LLM settings
LLM_MODEL = "gpt-5-nano"  # or your preferred model
LLM_TEMPERATURE = 0.7
LLM_REASONING_EFFORT = "high"  # "low", "medium", "high"
LLM_SUMMARY_MODE = None  # "auto", "detailed", or None

# UI Display Constraints (Word Limits)
MAX_DESCRIPTION_WORDS = 100
MAX_EVIDENCE_WORDS = 15
MAX_STORY_ALIGNMENT_WORDS = 30
MAX_PEER_BENCHMARKING_WORDS = 25

# Validation Settings
REQUIRE_SPECIFICITY = True
SPECIFICITY_THRESHOLD = 0.75  # 75% specificity score minimum
MAX_RETRY_ATTEMPTS = 2

# Debug settings
DEBUG_MODE = True
SHOW_FULL_PROMPTS = True

print("Configuration:")
print(f"  Use Decision Tree: {USE_DECISION_TREE}")
print(f"  Decision Tree Path: {DECISION_TREE_PATH}")
print(f"  LLM Model: {LLM_MODEL}")
print(f"  Temperature: {LLM_TEMPERATURE}")
print(f"  Debug Mode: {DEBUG_MODE}")
print(f"\nUI Constraints:")
print(f"  Max Description: {MAX_DESCRIPTION_WORDS} words")
print(f"  Max Evidence: {MAX_EVIDENCE_WORDS} words")
print(f"  Max Story Alignment: {MAX_STORY_ALIGNMENT_WORDS} words")
print(f"  Max Peer Benchmarking: {MAX_PEER_BENCHMARKING_WORDS} words")
print(f"\nValidation:")
print(f"  Require Specificity: {REQUIRE_SPECIFICITY}")
print(f"  Specificity Threshold: {SPECIFICITY_THRESHOLD}")
print(f"  Max Retry Attempts: {MAX_RETRY_ATTEMPTS}")

Configuration:
  Use Decision Tree: True
  Decision Tree Path: c:\projects\coc\idiscovery-deep-research\configs\decision_tree_rules.yaml
  LLM Model: gpt-5-nano
  Temperature: 0.7
  Debug Mode: True

UI Constraints:
  Max Description: 100 words
  Max Evidence: 15 words
  Max Story Alignment: 30 words
  Max Peer Benchmarking: 25 words

Validation:
  Require Specificity: True
  Specificity Threshold: 0.75
  Max Retry Attempts: 2


## 3. Load Input JSON Files

In this section, we load the pattern, reimbursement, and correlation JSON files that will be used to generate recommendations.

In [4]:
# ============= INPUT FILE PATHS (EDIT HERE) =============

# Pattern file (required)
PATTERN_FILE = "pattern_results_20260520_IP_AUTH-Commercial-202604-R3-202601.json"

# Reimbursement file (optional)
REIMBURSEMENT_FILE = "reimbursement_formatted_output_20260527_131443.json"

# Correlation file (optional - placeholder for future)
CORRELATION_FILE = None  # Set to filename when available

print("Input File Configuration:")
print(f"  Pattern File: {PATTERN_FILE}")
print(f"  Reimbursement File: {REIMBURSEMENT_FILE}")
print(f"  Correlation File: {CORRELATION_FILE or 'Not configured'}")

Input File Configuration:
  Pattern File: pattern_results_20260520_IP_AUTH-Commercial-202604-R3-202601.json
  Reimbursement File: reimbursement_formatted_output_20260527_131443.json
  Correlation File: Not configured


### 3.1 Load Pattern Data

In [5]:
# Load pattern data
pattern_data = None

if PATTERN_FILE:
    pattern_path = Path(PATTERN_FILE)
    if pattern_path.exists():
        with open(pattern_path, 'r', encoding='utf-8') as f:
            pattern_data = json.load(f)
        
        # Extract business patterns
        business_patterns = pattern_data.get('output', {}).get('business_patterns', [])
        cards = pattern_data.get('output', {}).get('cards', [])
        
        print(f"✓ Pattern data loaded from: {PATTERN_FILE}")
        print(f"  Job ID: {pattern_data.get('job_id', 'N/A')}")
        print(f"  Conversation ID: {pattern_data.get('conversation_id', 'N/A')}")
        print(f"  Status: {pattern_data.get('status', 'N/A')}")
        print(f"  Business Patterns: {len(business_patterns)}")
        print(f"  Cards: {len(cards)}")
        
        # Show first pattern summary
        if business_patterns:
            first_pattern = business_patterns[0]
            print(f"\n  First Pattern:")
            print(f"    Rank: {first_pattern.get('pattern_rank', 'N/A')}")
            print(f"    Title: {first_pattern.get('top_pattern', 'N/A')}")
            print(f"    Type: {first_pattern.get('pattern_type', 'N/A')}")
            print(f"    Impact: {first_pattern.get('impact_summary', {}).get('estimated_delta', 'N/A')}")
    else:
        print(f"❌ Pattern file not found: {PATTERN_FILE}")
else:
    print("⊗ No pattern file configured")

✓ Pattern data loaded from: pattern_results_20260520_IP_AUTH-Commercial-202604-R3-202601.json
  Job ID: fef01248fe1b475ab9163f19de048831
  Conversation ID: tutorial-IP_AUTH-Commercial-202604-R3-202601
  Status: success
  Business Patterns: 8
  Cards: 85

  First Pattern:
    Rank: 1
    Title: Authorization coding and facility mapping shifts are distorting the read
    Type: coding_mapping_validation
    Impact: ≈$19.5M affected


### 3.2 Load Reimbursement Data

In [6]:
# Load reimbursement data
reimbursement_data = None

if REIMBURSEMENT_FILE:
    reimb_path = Path(REIMBURSEMENT_FILE)
    if reimb_path.exists():
        with open(reimb_path, 'r', encoding='utf-8') as f:
            reimbursement_data = json.load(f)
        
        # Extract summary table and individual policies
        summary_table = reimbursement_data.get('summary_table', {})
        individual_policies = reimbursement_data.get('individual_policies', [])
        
        print(f"✓ Reimbursement data loaded from: {REIMBURSEMENT_FILE}")
        print(f"  Summary Table:")
        print(f"    Title: {summary_table.get('title', 'N/A')}")
        print(f"    Subtitle: {summary_table.get('subtitle', 'N/A')[:100]}...")
        print(f"    Payers: {len(summary_table.get('rows', []))}")
        print(f"    Columns: {len(summary_table.get('columns', []))}")
        print(f"  Individual Policies: {len(individual_policies)}")
        
        # Show first policy summary
        if individual_policies:
            first_policy = individual_policies[0]
            print(f"\n  First Policy:")
            print(f"    Payer: {first_policy.get('payer_name', 'N/A')}")
            print(f"    Title: {first_policy.get('policy_title', 'N/A')[:60]}...")
            print(f"    Tags: {', '.join(first_policy.get('tags', []))}")
            print(f"    Effective Date: {first_policy.get('effective_date', 'N/A')}")
    else:
        print(f"❌ Reimbursement file not found: {REIMBURSEMENT_FILE}")
else:
    print("⊗ No reimbursement file configured")

✓ Reimbursement data loaded from: reimbursement_formatted_output_20260527_131443.json
  Summary Table:
    Title: Payer Policy Summary
    Subtitle: Policy Analysis for CPT codes ["ECMO or Tracheostomy with Mechanical Ventilation >96 Hours or Princi...
    Payers: 6
    Columns: 10
  Individual Policies: 27

  First Policy:
    Payer: United Health
    Title: UnitedHealthcare Medicare Advantage Reimbursement Policy CMS...
    Tags: Medicare Advantage
    Effective Date: 03/01/2026


### 3.3 Load Correlation Data (Placeholder)

In [7]:
# Load correlation data (placeholder - not yet available)
correlation_data = None

if CORRELATION_FILE:
    corr_path = Path(CORRELATION_FILE)
    if corr_path.exists():
        with open(corr_path, 'r', encoding='utf-8') as f:
            correlation_data = json.load(f)
        
        print(f"✓ Correlation data loaded from: {CORRELATION_FILE}")
        # TODO: Add correlation data structure parsing when available
    else:
        print(f"❌ Correlation file not found: {CORRELATION_FILE}")
else:
    print("⊗ No correlation file configured (placeholder for future)")
    print("  Note: Correlation data will be integrated when available")

⊗ No correlation file configured (placeholder for future)
  Note: Correlation data will be integrated when available


### 3.4 Combine and Prepare Input Data

Combine pattern, reimbursement, and correlation data into a unified structure for recommendation generation.

In [8]:
# Combine data sources into unified input structure
input_data = []

if pattern_data and business_patterns:
    print("[COMBINING DATA] Building unified input from loaded sources...")
    print("="*80)
    
    # Process each business pattern
    for pattern in business_patterns:
        pattern_rank = pattern.get('pattern_rank', 0)
        pattern_title = pattern.get('top_pattern', '')
        pattern_type = pattern.get('pattern_type', '')
        service_category = pattern.get('what_is_impacting', '')
        
        # Extract evidence from pattern
        evidence_summary = pattern.get('evidence_summary', [])
        impact_summary = pattern.get('impact_summary', {})
        
        # Build claim evidence section
        claim_evidence = {
            "summary": pattern.get('pattern_details', ''),
            "details": evidence_summary,
            "impact": impact_summary.get('estimated_delta', 'N/A'),
            "direction": impact_summary.get('direction', 'unknown')
        }
        
        # Build reimbursement section if available
        reimbursement_section = {}
        if reimbursement_data and individual_policies:
            # Filter relevant policies based on pattern
            # For now, include all policies - can be refined later
            reimbursement_section = {
                "individual_policies": individual_policies[:10],  # Limit to first 10 for brevity
                "summary_table": summary_table
            }
        
        # Build correlation section if available
        correlation_section = {}
        if correlation_data:
            # TODO: Add correlation data when available
            correlation_section = correlation_data
        
        # Build unified pattern entry
        unified_entry = {
            "rank": pattern_rank,
            "pattern_title": pattern_title,
            "pattern_type": pattern_type,
            "pattern_description": pattern.get('pattern_details', ''),
            "service_category": service_category,
            "priority_entities": pattern.get('priority_entities', {}),
            "key_driver_codes": pattern.get('key_driver_codes', []),
            "why_it_matters": pattern.get('why_it_matters', ''),
            "recommended_next_step": pattern.get('recommended_next_step', ''),
            "validation_needed": pattern.get('validation_needed', False),
            "downstream_routes": pattern.get('downstream_routes', []),
            "explanation": {
                "claim_evidence": claim_evidence
            }
        }
        
        # Add reimbursement if available
        if reimbursement_section:
            unified_entry["explanation"]["reimbursement"] = reimbursement_section
        
        # Add correlation if available
        if correlation_section:
            unified_entry["explanation"]["correlation"] = correlation_section
        
        input_data.append(unified_entry)
    
    print(f"✓ Combined input data prepared: {len(input_data)} pattern(s)")
    print(f"\nPattern 1: {input_data[0]['pattern_title']}")
    print(f"  Service Category: {input_data[0]['service_category']}")
    
    # Check what data sources are included
    has_reimb = 'reimbursement' in input_data[0].get('explanation', {})
    has_corr = 'correlation' in input_data[0].get('explanation', {})
    print(f"  Has Reimbursement Data: {has_reimb}")
    print(f"  Has Correlation Data: {has_corr}")
    
    if len(input_data) > 1:
        print(f"\nPattern 2: {input_data[1]['pattern_title']}")
        print(f"  Service Category: {input_data[1]['service_category']}")
    
else:
    print("❌ No pattern data available to combine")
    print("  Please ensure pattern file is loaded correctly")

print("="*80)

[COMBINING DATA] Building unified input from loaded sources...
✓ Combined input data prepared: 8 pattern(s)

Pattern 1: Authorization coding and facility mapping shifts are distorting the read
  Service Category: Commercial inpatient authorization classification and facility mapping
  Has Reimbursement Data: True
  Has Correlation Data: False

Pattern 2: Commercial HMO acute hospital admissions surged in Colorado and Maine
  Service Category: Commercial HMO / Acute Hospital inpatient


### 3.5 Data Summary and Validation

In [9]:
# Display summary of loaded data
print("="*80)
print("DATA LOAD SUMMARY")
print("="*80)

print(f"\n📊 Pattern Data:")
print(f"   Loaded: {'✓ Yes' if pattern_data else '✗ No'}")
if pattern_data:
    print(f"   Patterns: {len(business_patterns)}")
    print(f"   Cards: {len(cards)}")

print(f"\n💰 Reimbursement Data:")
print(f"   Loaded: {'✓ Yes' if reimbursement_data else '✗ No'}")
if reimbursement_data:
    print(f"   Payers: {len(summary_table.get('rows', []))}")
    print(f"   Policies: {len(individual_policies)}")
    print(f"   Columns: {len(summary_table.get('columns', []))}")

print(f"\n🔗 Correlation Data:")
print(f"   Loaded: {'✓ Yes' if correlation_data else '✗ No'}")
if not correlation_data:
    print(f"   Status: Placeholder (not yet available)")

print(f"\n📋 Combined Input Data:")
print(f"   Ready: {'✓ Yes' if input_data else '✗ No'}")
if input_data:
    print(f"   Total Patterns: {len(input_data)}")
    for i, pattern in enumerate(input_data[:3], 1):  # Show first 3
        print(f"\n   Pattern {i}:")
        print(f"      Title: {pattern['pattern_title']}")
        print(f"      Type: {pattern['pattern_type']}")
        print(f"      Service: {pattern['service_category']}")
        has_reimb = 'reimbursement' in pattern.get('explanation', {})
        has_corr = 'correlation' in pattern.get('explanation', {})
        print(f"      Data: Claim Evidence ✓ | Reimbursement {'✓' if has_reimb else '✗'} | Correlation {'✓' if has_corr else '✗'}")
    
    if len(input_data) > 3:
        print(f"\n   ... and {len(input_data) - 3} more pattern(s)")

print("\n" + "="*80)

DATA LOAD SUMMARY

📊 Pattern Data:
   Loaded: ✓ Yes
   Patterns: 8
   Cards: 85

💰 Reimbursement Data:
   Loaded: ✓ Yes
   Payers: 6
   Policies: 27
   Columns: 10

🔗 Correlation Data:
   Loaded: ✗ No
   Status: Placeholder (not yet available)

📋 Combined Input Data:
   Ready: ✓ Yes
   Total Patterns: 8

   Pattern 1:
      Title: Authorization coding and facility mapping shifts are distorting the read
      Type: coding_mapping_validation
      Service: Commercial inpatient authorization classification and facility mapping
      Data: Claim Evidence ✓ | Reimbursement ✓ | Correlation ✗

   Pattern 2:
      Title: Commercial HMO acute hospital admissions surged in Colorado and Maine
      Type: mixed_volume_unit_cost
      Service: Commercial HMO / Acute Hospital inpatient
      Data: Claim Evidence ✓ | Reimbursement ✓ | Correlation ✗

   Pattern 3:
      Title: Provider concentration is raising acute hospital spend at Yale and Henrico
      Type: provider_network_economics
      Servic

In [10]:
# Validate and serialize input
def validate_json_input(input_data: Any) -> bool:
    """Validate that input is valid JSON."""
    if isinstance(input_data, (dict, list)):
        return True
    if isinstance(input_data, str):
        try:
            json.loads(input_data)
            return True
        except json.JSONDecodeError:
            return False
    return False

def serialize_input_for_llm(input_data: Any) -> str:
    """Convert input to JSON string for LLM processing."""
    if isinstance(input_data, str):
        parsed = json.loads(input_data)
        return json.dumps(parsed, indent=2, ensure_ascii=False)
    return json.dumps(input_data, indent=2, ensure_ascii=False)

# Validate input
if not validate_json_input(input_data):
    print("❌ Input validation failed")
else:
    print("✓ Input validation passed")
    
# Serialize for LLM
input_json_str = serialize_input_for_llm(input_data)
print(f"✓ Input serialized: {len(input_json_str)} characters")

if SHOW_FULL_PROMPTS:
    print("\n" + "="*80)
    print("SERIALIZED INPUT (first 500 chars):")
    print("="*80)
    print(input_json_str[:500] + "...")

✓ Input validation passed
✓ Input serialized: 154276 characters

SERIALIZED INPUT (first 500 chars):
[
  {
    "rank": 1,
    "pattern_title": "Authorization coding and facility mapping shifts are distorting the read",
    "pattern_type": "coding_mapping_validation",
    "pattern_description": "A large share of the apparent cost movement is tied to authorization code redistribution rather than a clean underlying utilization trend. In both Colorado and Maine, dollars moved sharply out of the negative or unknown authorization bucket while both PA-required and non-PA segments increased, and Colora...


## 4. Decision Tree Rules Section

In [11]:
# Load decision tree rules (if enabled)
rule_engine = None

if USE_DECISION_TREE:
    if DECISION_TREE_PATH.exists():
        try:
            rule_engine = DecisionTreeRuleEngine(str(DECISION_TREE_PATH))
            rule_count = rule_engine.get_total_rule_count()
            category_count = len(rule_engine.get_category_names())
            print(f"✓ Decision tree rules loaded")
            print(f"  Total rules: {rule_count}")
            print(f"  Categories: {category_count}")
        except Exception as e:
            print(f"❌ Failed to load decision tree: {e}")
    else:
        print(f"⚠ Decision tree YAML not found at: {DECISION_TREE_PATH}")
else:
    print("⊗ Decision tree disabled (USE_DECISION_TREE = False)")

2026-05-27 17:17:30,609 - deep_research_agents.decision_tree_rules - INFO - Loaded 950 rules from 18 categories
✓ Decision tree rules loaded
  Total rules: 950
  Categories: 18


In [12]:
# View decision tree structure
if rule_engine:
    print("DECISION TREE CATEGORIES:")
    print("="*80)
    
    for cat in rule_engine.get_category_names():
        rules = rule_engine.get_rules_by_category(cat)
        print(f"\n{cat}: {len(rules)} rule(s)")
        
        # Show first rule as example
        if rules and len(rules) > 0:
            first_rule = rules[0]
            print(f"  Example Rule:")
            print(f"    Trend ID: {first_rule.get('trend_id', 'N/A')}")
            if 'why' in first_rule:
                print(f"    Why: {first_rule['why']}")
            if 'cost_of_care_suggestions' in first_rule:
                print(f"    Suggestion: {first_rule['cost_of_care_suggestions']}")
else:
    print("No decision tree rules loaded")

DECISION TREE CATEGORIES:

IP MedSurg (DNE): 60 rule(s)
  Example Rule:
    Trend ID: 1
    Why: Is variance isolated to subset of providers, limiting research to both behavior and contractual changes?
    Suggestion: None

IP OB Dlvry Well NB (DNE): 60 rule(s)
  Example Rule:
    Trend ID: 1
    Why: Is variance isolated to subset of providers, limiting research to both behavior and contractual changes?
    Suggestion: None

IP NICU (DNE): 61 rule(s)
  Example Rule:
    Trend ID: 1
    Why: Is variance isolated to subset of providers, limiting research to both behavior and contractual changes?
    Suggestion: None

IP BH (DNE): 68 rule(s)
  Example Rule:
    Trend ID: 1
    Why: Is variance isolated to subset of providers, limiting research to both behavior and contractual changes?
    Suggestion: None

IP NF (DNE): 67 rule(s)
  Example Rule:
    Trend ID: 1
    Why: Is variance isolated to subset of providers, limiting research to both behavior and contractual changes?
    Suggestion

In [13]:
# Format rules for LLM (compact format)
rules_text = ""

if rule_engine:
    rules_text = rule_engine.format_rules_compact()
    print(f"✓ Rules formatted for LLM: {len(rules_text)} characters")
    
    if SHOW_FULL_PROMPTS:
        print("\n" + "="*80)
        print("FORMATTED RULES:")
        print("="*80)
        print(rules_text)
else:
    print("No rules to format")

✓ Rules formatted for LLM: 139598 characters

FORMATTED RULES:

IP MedSurg (DNE):
  - RESEARCH: Analyst to review Contract Provisions | WHY: Is variance explainable by contract changes or concessions that limit ELV management? | SUGGESTION: Contract remediation opportunity?  Alternative management options?
  - RESEARCH: Analyst to review Provider distribution | WHY: Is variance explainable by utilization increases at higher cost facilities? | SUGGESTION: Contract remediation opportunity?  Establish site of care opportunity?
  - RESEARCH: Analyst to review Contract Relativity | WHY: Is variance explainable by unit cost increases expected in UPTPM | SUGGESTION: Contract compliance concern?
  - RESEARCH: Analyst to review In-Network vs Out-of-Network status | WHY: Is variance explainable by increased distribution of OON services? | SUGGESTION: Address root cause of OON necessity?  Is the provider actually OON or do we have a claims payment issue?
  - RESEARCH: Analyst to review Provider D

## 5. UI-Optimized Prompts

In [14]:
# ============= UI-OPTIMIZED SYSTEM PROMPT =============

SYSTEM_PROMPT_UI = """You are a healthcare policy analyst generating recommendations for UI display.

CRITICAL UI CONSTRAINTS:
- Short, direct sentences
- Concrete identifiers: CPT codes, policy IDs, provider names, amounts
- Avoid: "Review", "Consider", "Explore", "Investigate"
- Use: "Implement", "Deny", "Restrict", "Update", "Renegotiate"

WORD LIMITS (STRICTLY ENFORCED):
- Description: MAX 100 words
- Evidence: MAX 15 words each
- Story alignment: MAX 30 words each
- Peer benchmarking: MAX 25 words each

STORY ALIGNMENT (FLEXIBLE PHRASING BUT MUST INCLUDE):
- WHY the business rule was selected (what pattern matched)
- HOW the recommendation addresses the situation (what action resolves it)
- Include: Rule reference + Situation explanation + Action relevance
- Example: "Trend 2.3 matches inappropriate critical care billing. Home discharge indicates non-admission. Denying CPT 99291 prevents $35M inappropriate payments."

SPECIFICITY REQUIREMENTS:
- Include: CPT/DRG codes, policy IDs, provider names, facility names, specific dollar amounts
- Be specific about WHAT to do, WHERE to apply, WHEN to implement
- GOOD: "Deny CPT 99291 when discharge status is 01 (Home) in TX, FL, CA effective Q3 2026"
- BAD: "Review critical care billing patterns and consider policy updates"

OUTPUT: Valid JSON only. No markdown, no explanations.
"""

print("✓ UI-optimized system prompt defined")
print(f"  Length: {len(SYSTEM_PROMPT_UI)} characters")

✓ UI-optimized system prompt defined
  Length: 1324 characters


In [15]:
# ============= UI-OPTIMIZED USER PROMPT =============

USER_PROMPT_UI_TEMPLATE = """Analyze healthcare data to generate policy recommendations for UI display.

INPUT DATA:
{input_json}

DECISION TREE RULES:
{decision_tree_rules}

INSTRUCTIONS:
1. Match input patterns to decision tree rules
2. Generate recommendations following UI constraints
3. For each recommendation:
   - Write SPECIFIC action with identifiers (codes, names, amounts)
   - Extract evidence (max 15 words each)
   - Explain story alignment (max 30 words each):
     * MUST include business rule explanation showing HOW it fits the context
     * Explain the situation and action
     * Format flexibly but ensure rule relevance is clear
   - Include peer benchmarking if available (max 25 words each)
   - Cite policy URLs from input data (format: "Policy: <PayerName> - <URL>")

STORY ALIGNMENT EXAMPLES (MUST explain how business rule fits):
✅ GOOD: "Business Rule Phys Emergency Trend 2.3 applies: inappropriate critical care billing with home discharge detected. Denying CPT 99291 prevents $35M inappropriate payments."
✅ GOOD: "Per IP MedSurg Trend 1.7 for provider cost outliers: 5 facilities show 40% excess costs. Contract renegotiation targets these high-cost providers."
✅ GOOD: "Matches Business Rule IP MedSurg Trend 1.7 on provider variance. Outlier facilities identified with $5.3K per-case excess. Renegotiate contracts with top 5 providers."
❌ BAD: "This recommendation addresses the pattern identified in the data." (No rule reference)
❌ BAD: "Based on analysis, action should be taken." (No rule, no specifics)

CITATION EXAMPLES (Policy URLs ONLY, NOT business rules):
✅ GOOD: "Policy: Molina Medicaid - https://example.com/molina_critical_care_policy.pdf"
✅ GOOD: "Policy: UnitedHealthcare - https://example.com/uhc_critical_care_policy.pdf"
❌ BAD: "Business Rule: Phys Emergency (IP, OP) - Trend 2.3" (Business rules go in story_alignment)
❌ BAD: "Decision Tree Category: IP MedSurg" (Business rules go in story_alignment)

SPECIFICITY EXAMPLES:
✅ GOOD: "Deny CPT 99291 when discharge status is 01 (Home) in TX, FL, CA effective Q3 2026."
✅ GOOD: "Renegotiate contracts with Providers A, B, C showing $18.5K avg cost vs $13.2K network avg."
❌ BAD: "Review critical care billing patterns and consider policy updates."
❌ BAD: "Explore opportunities to reduce costs at high-cost facilities."

OUTPUT SCHEMA:
{{
  "recommendations": [
    {{
      "rank": <number>,
      "priority": "HIGH|MEDIUM|LOW",
      "category": "Policy",
      "description": "<specific action with identifiers, max 100 words>",
      "evidence": ["<max 15 words>", "<max 15 words>"],
      "story_alignment": ["<business rule + how it fits + situation + action, max 30 words>"],
      "peer_benchmarking": ["<peer payor: practice, max 25 words>"],
      "citation": ["Policy: <PayerName> - <URL>"]
    }}
  ]
}}

CRITICAL:
- Strictly adhere to word count limits
- Story alignment MUST explain HOW the business rule fits the context
- Citations must be policy URLs ONLY (extract from input data's policy_url fields)
- If no policy URLs available in input, citations array can be empty
- Business rules belong in story_alignment, NOT in citations
- Be specific with identifiers (CPT codes, amounts, names)
- Use action verbs, avoid generic phrases
- NO markdown formatting, NO bullet points in list items, NO serial numbers

Return ONLY valid JSON.
"""

print("✓ UI-optimized user prompt template defined")
print(f"  Length: {len(USER_PROMPT_UI_TEMPLATE)} characters")

✓ UI-optimized user prompt template defined
  Length: 3328 characters


## 6. Validation Functions

In [16]:
# ============= VALIDATION FUNCTIONS =============

from collections import Counter
from typing import Tuple

def count_words(text: str) -> int:
    """Count words in text."""
    return len(text.split())

def validate_word_counts(recommendations: List[Dict]) -> Dict:
    """Validate word count limits for all fields."""
    results = {
        "description_ok": [],
        "evidence_ok": [],
        "story_alignment_ok": [],
        "peer_benchmarking_ok": [],
        "violations": []
    }
    
    for i, rec in enumerate(recommendations):
        rank = rec.get("rank", i+1)
        
        # Description
        desc = rec.get("description", "")
        desc_words = count_words(desc)
        desc_ok = desc_words <= MAX_DESCRIPTION_WORDS
        results["description_ok"].append(desc_ok)
        if not desc_ok:
            results["violations"].append(f"Rec {rank}: Description {desc_words} words (max {MAX_DESCRIPTION_WORDS})")
        
        # Evidence
        evidence = rec.get("evidence", [])
        evidence_ok = all(count_words(e) <= MAX_EVIDENCE_WORDS for e in evidence)
        results["evidence_ok"].append(evidence_ok)
        for j, e in enumerate(evidence):
            e_words = count_words(e)
            if e_words > MAX_EVIDENCE_WORDS:
                results["violations"].append(f"Rec {rank}: Evidence[{j}] {e_words} words (max {MAX_EVIDENCE_WORDS})")
        
        # Story alignment
        story = rec.get("story_alignment", [])
        story_ok = all(count_words(s) <= MAX_STORY_ALIGNMENT_WORDS for s in story)
        results["story_alignment_ok"].append(story_ok)
        for j, s in enumerate(story):
            s_words = count_words(s)
            if s_words > MAX_STORY_ALIGNMENT_WORDS:
                results["violations"].append(f"Rec {rank}: Story[{j}] {s_words} words (max {MAX_STORY_ALIGNMENT_WORDS})")
        
        # Peer benchmarking
        peer = rec.get("peer_benchmarking", [])
        peer_ok = all(count_words(p) <= MAX_PEER_BENCHMARKING_WORDS for p in peer)
        results["peer_benchmarking_ok"].append(peer_ok)
        for j, p in enumerate(peer):
            p_words = count_words(p)
            if p_words > MAX_PEER_BENCHMARKING_WORDS:
                results["violations"].append(f"Rec {rank}: Peer[{j}] {p_words} words (max {MAX_PEER_BENCHMARKING_WORDS})")
    
    return results

def validate_story_alignment(recommendations: List[Dict]) -> Dict:
    """Validate story alignment explains rule selection and relevance."""
    results = {
        "has_business_rule": [],
        "has_rule_fit_explanation": [],
        "has_situation_explanation": [],
        "has_action_relevance": [],
        "quality_score": [],
        "issues": []
    }
    
    for i, rec in enumerate(recommendations):
        rank = rec.get("rank", i+1)
        story_items = rec.get("story_alignment", [])
        
        if not story_items:
            results["has_business_rule"].append(False)
            results["has_rule_fit_explanation"].append(False)
            results["has_situation_explanation"].append(False)
            results["has_action_relevance"].append(False)
            results["quality_score"].append(0.0)
            results["issues"].append(f"Rec {rank}: No story alignment provided")
            continue
        
        # Combine all story items for analysis
        combined_story = " ".join(story_items).lower()
        
        # Check for business rule reference
        business_rule_keywords = ["business rule", "trend", "per ip", "per op", "per phys", "per retail", "matches business"]
        has_business_rule = any(kw in combined_story for kw in business_rule_keywords)
        results["has_business_rule"].append(has_business_rule)
        if not has_business_rule:
            results["issues"].append(f"Rec {rank}: Missing business rule reference")
        
        # Check for rule fit explanation (how rule applies to context)
        fit_keywords = ["applies", "fits", "matches", "relevant because", "addresses", "for", "on", "aligns with"]
        has_fit = any(kw in combined_story for kw in fit_keywords) and has_business_rule
        results["has_rule_fit_explanation"].append(has_fit)
        if not has_fit and has_business_rule:
            results["issues"].append(f"Rec {rank}: Missing explanation of HOW rule fits context")
        
        # Check for situation explanation
        situation_keywords = ["pattern", "shows", "indicates", "identified", "found", "detected", "because", "variance", "inappropriate", "outlier"]
        has_situation = any(kw in combined_story for kw in situation_keywords)
        results["has_situation_explanation"].append(has_situation)
        if not has_situation:
            results["issues"].append(f"Rec {rank}: Missing situation explanation")
        
        # Check for action relevance
        action_keywords = ["deny", "restrict", "implement", "update", "renegotiate", "prevents", "addresses", "targets", "resolves", "reduces"]
        has_action = any(kw in combined_story for kw in action_keywords)
        results["has_action_relevance"].append(has_action)
        if not has_action:
            results["issues"].append(f"Rec {rank}: Missing action relevance")
        
        # Calculate quality score
        quality = (has_business_rule + has_fit + has_situation + has_action) / 4.0
        results["quality_score"].append(quality)
    
    return results

def validate_specificity(recommendations: List[Dict]) -> Dict:
    """Validate recommendations have concrete identifiers."""
    results = {
        "has_identifiers": [],
        "avoids_generic": [],
        "uses_action_verbs": [],
        "specificity_score": [],
        "issues": []
    }
    
    generic_phrases = ["review", "consider", "explore", "investigate", "assess", "evaluate", "analyze"]
    action_verbs = ["implement", "deny", "restrict", "update", "apply", "renegotiate", "remove", "add", "establish", "require"]
    
    for i, rec in enumerate(recommendations):
        rank = rec.get("rank", i+1)
        description = rec.get("description", "").lower()
        
        # Check for identifiers
        has_cpt = bool(re.search(r'\\bcpt\\s*\\d{5}\\b|\\b\\d{5}\\b', description))
        has_drg = bool(re.search(r'\\bdrg[\\s-]?\\d+\\b', description))
        has_amount = bool(re.search(r'\\$[\\d,]+[kmb]?', description))
        has_provider = bool(re.search(r'\\bprovider[s]?\\b|\\bfacility\\b|\\bhospital\\b|\\bphysician\\b', description))
        
        has_identifiers = has_cpt or has_drg or has_amount or has_provider
        results["has_identifiers"].append(has_identifiers)
        if not has_identifiers:
            results["issues"].append(f"Rec {rank}: Missing concrete identifiers (CPT/DRG/amounts/providers)")
        
        # Check for generic phrases
        found_generic = [g for g in generic_phrases if g in description]
        avoids_generic = len(found_generic) == 0
        results["avoids_generic"].append(avoids_generic)
        if not avoids_generic:
            results["issues"].append(f"Rec {rank}: Uses generic phrases: {found_generic}")
        
        # Check for action verbs
        found_action = [a for a in action_verbs if a in description]
        uses_action_verbs = len(found_action) > 0
        results["uses_action_verbs"].append(uses_action_verbs)
        if not uses_action_verbs:
            results["issues"].append(f"Rec {rank}: Missing action verbs")
        
        # Calculate specificity score
        score = sum([has_cpt, has_drg, has_amount, has_provider, uses_action_verbs, avoids_generic]) / 6.0
        results["specificity_score"].append(score)
    
    return results

def validate_citations(recommendations: List[Dict]) -> Dict:
    """Validate citations contain policy URLs, not business rules."""
    results = {
        "has_policy_urls": [],
        "avoids_business_rules": [],
        "issues": []
    }
    
    for i, rec in enumerate(recommendations):
        rank = rec.get("rank", i+1)
        citations = rec.get("citation", [])
        
        if not citations:
            # Empty citations are OK if no policy URLs in input
            results["has_policy_urls"].append(True)
            results["avoids_business_rules"].append(True)
            continue
        
        combined_citations = " ".join(citations).lower()
        
        # Check for policy URLs
        has_url = bool(re.search(r'https?://', combined_citations))
        has_policy_prefix = "policy:" in combined_citations or "policy " in combined_citations
        has_policy_urls = has_url and has_policy_prefix
        results["has_policy_urls"].append(has_policy_urls)
        if not has_policy_urls:
            results["issues"].append(f"Rec {rank}: Missing policy URLs in citations (should be 'Policy: <Name> - <URL>')")
        
        # Check NOT business rules
        has_business_rule = "business rule" in combined_citations or ("trend" in combined_citations and not has_url)
        avoids_business_rules = not has_business_rule
        results["avoids_business_rules"].append(avoids_business_rules)
        if not avoids_business_rules:
            results["issues"].append(f"Rec {rank}: Citations contain business rules (move to story_alignment)")
    
    return results

print("✓ Validation functions defined")
print("  - validate_word_counts()")
print("  - validate_story_alignment() [UPDATED: checks business rule fit]")
print("  - validate_specificity()")
print("  - validate_citations() [NEW: validates policy URLs]")

✓ Validation functions defined
  - validate_word_counts()
  - validate_story_alignment() [UPDATED: checks business rule fit]
  - validate_specificity()
  - validate_citations() [NEW: validates policy URLs]


## 7. LLM Invocation with Retry Logic

In [17]:
# ============= LLM RETRY FUNCTION =============

def invoke_llm_with_retry(
    llm: Any,
    system_prompt: str,
    user_prompt: str,
    max_attempts: int = MAX_RETRY_ATTEMPTS
) -> Tuple[Dict, List[str]]:
    """Invoke LLM with validation retry logic.
    
    Returns:
        Tuple of (recommendations_dict, attempt_log)
    """
    attempt_log = []
    
    for attempt in range(1, max_attempts + 1):
        print(f"\\n[ATTEMPT {attempt}/{max_attempts}] Invoking LLM...")
        attempt_log.append(f"Attempt {attempt}:")
        
        # Build messages
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
        
        try:
            # Invoke LLM
            response = llm.invoke(messages)
            content = response.content.strip()
            
            # Remove markdown fences
            if content.startswith("```"):
                lines = content.split("\\n")
                if len(lines) > 2:
                    content = "\\n".join(lines[1:-1])
            
            # Parse JSON
            result = json.loads(content)
            recommendations = result.get("recommendations", [])
            
            print(f"[ATTEMPT {attempt}] ✓ Received {len(recommendations)} recommendation(s)")
            attempt_log.append(f"  Received {len(recommendations)} recommendations")
            
            # Validate
            print(f"[ATTEMPT {attempt}] Validating output...")
            
            word_count_results = validate_word_counts(recommendations)
            story_results = validate_story_alignment(recommendations)
            specificity_results = validate_specificity(recommendations)
            citation_results = validate_citations(recommendations)
            
            # Check if validation passed
            word_count_pass = len(word_count_results["violations"]) == 0
            story_pass = len(story_results["issues"]) == 0
            specificity_pass = all(score >= SPECIFICITY_THRESHOLD for score in specificity_results["specificity_score"])
            citation_pass = len(citation_results["issues"]) == 0
            
            validation_passed = word_count_pass and story_pass and specificity_pass and citation_pass
            
            if validation_passed:
                print(f"[ATTEMPT {attempt}] ✓ Validation PASSED")
                attempt_log.append(f"  Validation: PASSED")
                return result, attempt_log
            
            # Validation failed - prepare feedback for retry
            print(f"[ATTEMPT {attempt}] ⚠ Validation FAILED")
            attempt_log.append(f"  Validation: FAILED")
            
            feedback_parts = []
            
            if not word_count_pass:
                feedback_parts.append(f"Word count violations ({len(word_count_results['violations'])}):") 
                feedback_parts.extend([f"  - {v}" for v in word_count_results["violations"][:5]])
                attempt_log.append(f"  - Word count: {len(word_count_results['violations'])} violations")
            
            if not story_pass:
                feedback_parts.append(f"Story alignment issues ({len(story_results['issues'])}):") 
                feedback_parts.extend([f"  - {i}" for i in story_results["issues"][:5]])
                attempt_log.append(f"  - Story alignment: {len(story_results['issues'])} issues")
            
            if not specificity_pass:
                failed_count = sum(1 for score in specificity_results["specificity_score"] if score < SPECIFICITY_THRESHOLD)
                feedback_parts.append(f"Specificity issues ({failed_count} recommendations below threshold):") 
                feedback_parts.extend([f"  - {i}" for i in specificity_results["issues"][:5]])
                attempt_log.append(f"  - Specificity: {failed_count} below threshold")
            
            if not citation_pass:
                feedback_parts.append(f"Citation issues ({len(citation_results['issues'])}):") 
                feedback_parts.extend([f"  - {i}" for i in citation_results["issues"][:5]])
                attempt_log.append(f"  - Citations: {len(citation_results['issues'])} issues")
            
            feedback = "\\n".join(feedback_parts)
            print(f"[ATTEMPT {attempt}] Feedback:\\n{feedback}")
            
            if attempt < max_attempts:
                # Add feedback to user prompt for retry
                retry_prompt = f"{user_prompt}\\n\\nPREVIOUS ATTEMPT FEEDBACK:\\n{feedback}\\n\\nPlease fix these issues and try again."
                user_prompt = retry_prompt
            else:
                print(f"[ATTEMPT {attempt}] Max attempts reached, returning best effort")
                attempt_log.append(f"  Max attempts reached")
                return result, attempt_log
        
        except json.JSONDecodeError as e:
            print(f"[ATTEMPT {attempt}] ✗ JSON parse error: {e}")
            attempt_log.append(f"  JSON parse error: {str(e)}")
            if attempt == max_attempts:
                return {"recommendations": []}, attempt_log
        except Exception as e:
            print(f"[ATTEMPT {attempt}] ✗ Error: {e}")
            attempt_log.append(f"  Error: {str(e)}")
            if attempt == max_attempts:
                return {"recommendations": []}, attempt_log
    
    return {"recommendations": []}, attempt_log

print("✓ LLM retry function defined [UPDATED: includes citation validation]")

✓ LLM retry function defined [UPDATED: includes citation validation]


In [18]:
# Initialize LLM
from deep_research_core.base_agent import CredentialProvider, AgentBase

class TempAgent(AgentBase):
    def __init__(self):
        super().__init__(
            agent_name="temp_recommendation_agent",
            state_class=dict,
            llm_reasoning_effort=LLM_REASONING_EFFORT
        )
    
    @property
    def node_name(self):
        return "recommendation"
    
    def node_function(self, state):
        return state
    
    def extract_result(self, final_state):
        return final_state

print("[LLM] Initializing LLM client...")
temp_agent = TempAgent()
llm = temp_agent.llm
print("[LLM] ✓ LLM client initialized")

[LLM] Initializing LLM client...
2026-05-27 17:17:44,547 - deep_research_utils.ehap - INFO - Requesting new access token from https://api.horizon.elevancehealth.com/v2/oauth2/token with client_id: iYfcOHQHPTaAn3BckTey49nJg93QsBmE


c:\projects\coc\idiscovery-deep-research\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.horizon.elevancehealth.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


2026-05-27 17:17:45,564 - deep_research_utils.ehap - INFO - Access token generated successfully.
**TOKEN** **TOKEN** **TOKEN** 
[LLM] ✓ LLM client initialized


In [19]:
# Build prompts and invoke LLM with retry
print("="*80)
print("EXECUTING LLM WITH RETRY LOGIC")
print("="*80)

# Build user prompt
user_prompt = USER_PROMPT_UI_TEMPLATE.format(
    input_json=input_json_str,
    decision_tree_rules=rules_text if rules_text else "No decision tree rules loaded"
)

if SHOW_FULL_PROMPTS:
    print("\\n[PROMPT] System Prompt:")
    print("-"*80)
    print(SYSTEM_PROMPT_UI)
    print("-"*80)
    print("\\n[PROMPT] User Prompt (first 1000 chars):")
    print("-"*80)
    print(user_prompt[:1000] + "...")
    print("-"*80)

# Invoke with retry
recommendations_output, attempt_log = invoke_llm_with_retry(
    llm=llm,
    system_prompt=SYSTEM_PROMPT_UI,
    user_prompt=user_prompt
)

recommendations = recommendations_output.get("recommendations", [])

print("\\n" + "="*80)
print(f"EXECUTION COMPLETE: {len(recommendations)} recommendation(s) generated")
print("="*80)

print("\\nAttempt Log:")
for log_entry in attempt_log:
    print(log_entry)

EXECUTING LLM WITH RETRY LOGIC
\n[PROMPT] System Prompt:
--------------------------------------------------------------------------------
You are a healthcare policy analyst generating recommendations for UI display.

CRITICAL UI CONSTRAINTS:
- Short, direct sentences
- Concrete identifiers: CPT codes, policy IDs, provider names, amounts
- Avoid: "Review", "Consider", "Explore", "Investigate"
- Use: "Implement", "Deny", "Restrict", "Update", "Renegotiate"

WORD LIMITS (STRICTLY ENFORCED):
- Description: MAX 100 words
- Evidence: MAX 15 words each
- Story alignment: MAX 30 words each
- Peer benchmarking: MAX 25 words each

STORY ALIGNMENT (FLEXIBLE PHRASING BUT MUST INCLUDE):
- WHY the business rule was selected (what pattern matched)
- HOW the recommendation addresses the situation (what action resolves it)
- Include: Rule reference + Situation explanation + Action relevance
- Example: "Trend 2.3 matches inappropriate critical care billing. Home discharge indicates non-admission. Denyi

## 8. Quality Metrics Dashboard

In [20]:
# ============= QUALITY METRICS DASHBOARD =============

if recommendations:
    print("="*80)
    print("QUALITY METRICS DASHBOARD")
    print("="*80)
    
    # Run all validations
    word_count_results = validate_word_counts(recommendations)
    story_results = validate_story_alignment(recommendations)
    specificity_results = validate_specificity(recommendations)
    citation_results = validate_citations(recommendations)
    
    total_recs = len(recommendations)
    
    print(f"\\n📊 OVERVIEW")
    print(f"  Total Recommendations: {total_recs}")
    
    # Priority distribution
    priority_counts = Counter(rec.get("priority", "UNKNOWN") for rec in recommendations)
    print(f"\\n  Priority Distribution:")
    for priority in ["HIGH", "MEDIUM", "LOW"]:
        count = priority_counts.get(priority, 0)
        print(f"    - {priority}: {count}")
    
    # Word Count Compliance
    print(f"\\n📏 WORD COUNT COMPLIANCE")
    desc_pass = sum(word_count_results["description_ok"])
    evidence_pass = sum(word_count_results["evidence_ok"])
    story_pass = sum(word_count_results["story_alignment_ok"])
    peer_pass = sum(word_count_results["peer_benchmarking_ok"])
    
    print(f"  Description (max {MAX_DESCRIPTION_WORDS} words): {desc_pass}/{total_recs} ({100*desc_pass/total_recs:.0f}%)")
    print(f"  Evidence (max {MAX_EVIDENCE_WORDS} words): {evidence_pass}/{total_recs} ({100*evidence_pass/total_recs:.0f}%)")
    print(f"  Story Alignment (max {MAX_STORY_ALIGNMENT_WORDS} words): {story_pass}/{total_recs} ({100*story_pass/total_recs:.0f}%)")
    print(f"  Peer Benchmarking (max {MAX_PEER_BENCHMARKING_WORDS} words): {peer_pass}/{total_recs} ({100*peer_pass/total_recs:.0f}%)")
    
    if word_count_results["violations"]:
        print(f"\\n  ⚠ Word Count Violations:")
        for violation in word_count_results["violations"][:5]:
            print(f"    - {violation}")
    
    # Story Alignment Quality
    print(f"\\n📖 STORY ALIGNMENT QUALITY")
    business_rule_pass = sum(story_results["has_business_rule"])
    rule_fit_pass = sum(story_results["has_rule_fit_explanation"])
    situation_pass = sum(story_results["has_situation_explanation"])
    action_pass = sum(story_results["has_action_relevance"])
    avg_quality = sum(story_results["quality_score"]) / total_recs if total_recs > 0 else 0
    
    print(f"  Has Business Rule Reference: {business_rule_pass}/{total_recs} ({100*business_rule_pass/total_recs:.0f}%)")
    print(f"  Explains HOW Rule Fits Context: {rule_fit_pass}/{total_recs} ({100*rule_fit_pass/total_recs:.0f}%)")
    print(f"  Has Situation Explanation: {situation_pass}/{total_recs} ({100*situation_pass/total_recs:.0f}%)")
    print(f"  Has Action Relevance: {action_pass}/{total_recs} ({100*action_pass/total_recs:.0f}%)")
    print(f"  Average Quality Score: {avg_quality:.2f} / 1.00")
    
    if story_results["issues"]:
        print(f"\\n  ⚠ Story Alignment Issues:")
        for issue in story_results["issues"][:5]:
            print(f"    - {issue}")
    
    # Specificity Analysis
    print(f"\\n🎯 SPECIFICITY ANALYSIS")
    has_ids_pass = sum(specificity_results["has_identifiers"])
    avoids_generic_pass = sum(specificity_results["avoids_generic"])
    has_action_pass = sum(specificity_results["uses_action_verbs"])
    avg_specificity = sum(specificity_results["specificity_score"]) / total_recs if total_recs > 0 else 0
    above_threshold = sum(1 for score in specificity_results["specificity_score"] if score >= SPECIFICITY_THRESHOLD)
    
    print(f"  Has Concrete Identifiers: {has_ids_pass}/{total_recs} ({100*has_ids_pass/total_recs:.0f}%)")
    print(f"  Avoids Generic Phrases: {avoids_generic_pass}/{total_recs} ({100*avoids_generic_pass/total_recs:.0f}%)")
    print(f"  Uses Action Verbs: {has_action_pass}/{total_recs} ({100*has_action_pass/total_recs:.0f}%)")
    print(f"  Average Specificity Score: {avg_specificity:.2f} / 1.00")
    print(f"  Above Threshold ({SPECIFICITY_THRESHOLD}): {above_threshold}/{total_recs} ({100*above_threshold/total_recs:.0f}%)")
    
    if specificity_results["issues"]:
        print(f"\\n  ⚠ Specificity Issues:")
        for issue in specificity_results["issues"][:5]:
            print(f"    - {issue}")
    
    # Citation Validation (NEW)
    print(f"\\n📎 CITATION VALIDATION")
    has_urls_pass = sum(citation_results["has_policy_urls"])
    avoids_rules_pass = sum(citation_results["avoids_business_rules"])
    
    print(f"  Has Policy URLs: {has_urls_pass}/{total_recs} ({100*has_urls_pass/total_recs:.0f}%)")
    print(f"  Avoids Business Rules: {avoids_rules_pass}/{total_recs} ({100*avoids_rules_pass/total_recs:.0f}%)")
    
    if citation_results["issues"]:
        print(f"\\n  ⚠ Citation Issues:")
        for issue in citation_results["issues"][:5]:
            print(f"    - {issue}")
    
    # Overall Quality Score
    print(f"\\n⭐ OVERALL QUALITY SCORE")
    citation_score = (has_urls_pass + avoids_rules_pass) / (2 * total_recs) if total_recs > 0 else 0
    overall_score = (avg_quality + avg_specificity + citation_score +
                    (desc_pass + evidence_pass + story_pass + peer_pass) / (4 * total_recs)) / 4
    print(f"  Overall Score: {overall_score:.2f} / 1.00 ({100*overall_score:.0f}%)")
    
    print("\\n" + "="*80)
else:
    print("⚠ No recommendations generated, skipping quality metrics")

QUALITY METRICS DASHBOARD
\n📊 OVERVIEW
  Total Recommendations: 8
\n  Priority Distribution:
    - HIGH: 4
    - MEDIUM: 3
    - LOW: 1
\n📏 WORD COUNT COMPLIANCE
  Description (max 100 words): 8/8 (100%)
  Evidence (max 15 words): 8/8 (100%)
  Story Alignment (max 30 words): 8/8 (100%)
  Peer Benchmarking (max 25 words): 8/8 (100%)
\n📖 STORY ALIGNMENT QUALITY
  Has Business Rule Reference: 1/8 (12%)
  Explains HOW Rule Fits Context: 1/8 (12%)
  Has Situation Explanation: 4/8 (50%)
  Has Action Relevance: 7/8 (88%)
  Average Quality Score: 0.41 / 1.00
\n  ⚠ Story Alignment Issues:
    - Rec 1: Missing business rule reference
    - Rec 1: Missing situation explanation
    - Rec 1: Missing action relevance
    - Rec 2: Missing business rule reference
    - Rec 3: Missing business rule reference
\n🎯 SPECIFICITY ANALYSIS
  Has Concrete Identifiers: 0/8 (0%)
  Avoids Generic Phrases: 8/8 (100%)
  Uses Action Verbs: 8/8 (100%)
  Average Specificity Score: 0.33 / 1.00
  Above Threshold (0.75):

## 9. Display Recommendations with Word Counts

In [21]:
# Display recommendations in detail with word counts
if recommendations:
    print("="*80)
    print("RECOMMENDATIONS DETAIL")
    print("="*80)
    
    for i, rec in enumerate(recommendations):
        rank = rec.get("rank", i+1)
        priority = rec.get("priority", "UNKNOWN")
        description = rec.get("description", "")
        evidence = rec.get("evidence", [])
        story = rec.get("story_alignment", [])
        peer = rec.get("peer_benchmarking", [])
        citation = rec.get("citation", [])
        
        print(f"\\n[{rank}] PRIORITY: {priority}")
        print("-"*80)
        print(f"Description ({count_words(description)} words):") 
        print(f"  {description}")
        
        print(f"\\nEvidence ({len(evidence)} items):")
        for j, e in enumerate(evidence, 1):
            print(f"  {j}. {e} ({count_words(e)} words)")
        
        print(f"\\nStory Alignment ({len(story)} items):")
        for j, s in enumerate(story, 1):
            print(f"  {j}. {s} ({count_words(s)} words)")
        
        if peer:
            print(f"\\nPeer Benchmarking ({len(peer)} items):")
            for j, p in enumerate(peer, 1):
                print(f"  {j}. {p} ({count_words(p)} words)")
        
        if citation:
            print(f"\\nCitation:")
            for c in citation:
                print(f"  - {c}")
    
    print("\\n" + "="*80)
else:
    print("⚠ No recommendations to display")

RECOMMENDATIONS DETAIL
\n[1] PRIORITY: HIGH
--------------------------------------------------------------------------------
Description (66 words):
  Update AUTH_CODE_DISTRIBUTION and facility crosswalks for CO and ME HMO inpatient claims. Reclassify auth code -3 to PA Y or PA N only with matched authorization IDs. Map facility type Not Mapped claims, including 10 CO admissions at $145.8K each, before Q3 2026 UM, contract, or denial changes. Freeze provider actions for EMORY HILLANDALE HOSPITAL, EMORY DECATUR HOSPITAL, and HENRICO DOCTORS' HOSPITAL until corrected baselines load.
\nEvidence (3 items):
  1. CO auth movement: PA Y +$19.3M, PA N +$9.9M, -3 bucket -$28.0M. (12 words)
  2. ME shows same shift: PA Y +$10.6M, PA N +$3.9M, -3 -$8.3M. (12 words)
  3. CO Not Mapped added $1.5M across 10 admissions at $145.8K each. (11 words)
\nStory Alignment (2 items):
  1. IP MedSurg Auth to Claim match fits CO and ME PA Y, PA N, -3 shifts. Updating mapping restores valid UM actions. (22 word

## 10. Export Results with Quality Metrics

In [22]:
# Export results to JSON with quality metrics
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"recommendation_ui_optimized_{timestamp}.json"

# Build export data
export_data = {
    "metadata": {
        "timestamp": timestamp,
        "input_patterns": len(input_data),
        "decision_tree_used": USE_DECISION_TREE,
        "llm_model": LLM_MODEL,
        "ui_constraints": {
            "max_description_words": MAX_DESCRIPTION_WORDS,
            "max_evidence_words": MAX_EVIDENCE_WORDS,
            "max_story_alignment_words": MAX_STORY_ALIGNMENT_WORDS,
            "max_peer_benchmarking_words": MAX_PEER_BENCHMARKING_WORDS
        },
        "validation": {
            "specificity_threshold": SPECIFICITY_THRESHOLD,
            "max_retry_attempts": MAX_RETRY_ATTEMPTS
        },
        "attempt_log": attempt_log
    },
    "recommendations": recommendations,
    "quality_metrics": {} if not recommendations else {
        "word_count_compliance": {
            "description_pass_rate": sum(word_count_results["description_ok"]) / len(recommendations),
            "evidence_pass_rate": sum(word_count_results["evidence_ok"]) / len(recommendations),
            "story_alignment_pass_rate": sum(word_count_results["story_alignment_ok"]) / len(recommendations),
            "peer_benchmarking_pass_rate": sum(word_count_results["peer_benchmarking_ok"]) / len(recommendations)
        },
        "story_alignment_quality": {
            "average_quality_score": sum(story_results["quality_score"]) / len(recommendations),
            "has_business_rule_rate": sum(story_results["has_business_rule"]) / len(recommendations),
            "has_rule_fit_explanation_rate": sum(story_results["has_rule_fit_explanation"]) / len(recommendations),
            "has_situation_rate": sum(story_results["has_situation_explanation"]) / len(recommendations),
            "has_action_rate": sum(story_results["has_action_relevance"]) / len(recommendations)
        },
        "specificity": {
            "average_specificity_score": sum(specificity_results["specificity_score"]) / len(recommendations),
            "above_threshold_rate": sum(1 for s in specificity_results["specificity_score"] if s >= SPECIFICITY_THRESHOLD) / len(recommendations),
            "has_identifiers_rate": sum(specificity_results["has_identifiers"]) / len(recommendations),
            "avoids_generic_rate": sum(specificity_results["avoids_generic"]) / len(recommendations),
            "uses_action_verbs_rate": sum(specificity_results["uses_action_verbs"]) / len(recommendations)
        },
        "citations": {
            "has_policy_urls_rate": sum(citation_results["has_policy_urls"]) / len(recommendations),
            "avoids_business_rules_rate": sum(citation_results["avoids_business_rules"]) / len(recommendations)
        }
    }
}

with open(output_file, 'w') as f:
    json.dump(export_data, f, indent=2)

print(f"✓ Results exported to: {output_file}")
print(f"  Total recommendations: {len(recommendations)}")
print(f"  File size: {Path(output_file).stat().st_size:,} bytes")

✓ Results exported to: recommendation_ui_optimized_20260527_172638.json
  Total recommendations: 8
  File size: 14,583 bytes


## Summary

**UI-Optimized Recommendation Agent - Complete Workflow**

This notebook successfully implements UI-optimized recommendation generation:

✅ **Story Alignment with Business Rules**: Each recommendation explains HOW the business rule fits the context
- Must include business rule reference (category + trend)
- Must explain HOW the rule fits the situation
- Must describe the situation and action
- Flexible phrasing but clear relevance
- Example: "Business Rule Phys Emergency Trend 2.3 applies: inappropriate critical care billing with home discharge detected. Denying CPT 99291 prevents $35M inappropriate payments."

✅ **Citations → Policy URLs**: Citations now contain policy document links, NOT business rules
- Format: "Policy: <PayerName> - <URL>"
- Business rules moved to story_alignment
- Empty citations OK if no policy URLs in input
- Example: "Policy: Molina Medicaid - https://example.com/molina_critical_care_policy.pdf"

✅ **Specificity**: Concrete identifiers required
- CPT/DRG codes
- Policy IDs
- Dollar amounts
- Provider/facility names

✅ **Brevity**: Strict word count limits
- Description: 100 words max
- Evidence: 15 words max per bullet
- Story Alignment: 30 words max per bullet
- Peer Benchmarking: 25 words max per bullet

✅ **Validation with Retry**: Automatic retry with feedback
- Max 2 attempts
- Validates: word counts, story alignment, specificity, citations
- Detailed feedback on violations
- Quality score tracking

✅ **Quality Metrics**: Comprehensive dashboard
- Word count compliance
- Story alignment quality (business rule/fit/situation/action)
- Specificity score (identifiers/action verbs)
- Citation validation (policy URLs/no business rules)
- Overall quality score

## Key Changes from Original

1. **Citations Changed**: From business decision tree rules → Policy URLs from input data
2. **Story Alignment Enhanced**: Now MUST include business rule + HOW it fits context
3. **Validation Updated**: Added citation validation function
4. **Quality Dashboard**: Added citation compliance metrics
5. **Retry Logic**: Includes citation validation in retry feedback

## Expected Output Structure

```json
{
  "rank": 1,
  "priority": "HIGH",
  "description": "Deny CPT 99291 when discharge status is 01 (Home) in TX, FL, CA markets effective Q3 2026.",
  "evidence": [
    "$35.1M total exposure",
    "High volume home discharge"
  ],
  "story_alignment": [
    "Business Rule Phys Emergency Trend 2.3 applies: inappropriate critical care billing with home discharge detected. Denying CPT 99291 prevents $35M inappropriate payments."
  ],
  "peer_benchmarking": [
    "Molina: Denies critical care in ED for home discharge since 2024"
  ],
  "citation": [
    "Policy: Molina Medicaid - https://example.com/molina_critical_care_policy.pdf",
    "Policy: UnitedHealthcare - https://example.com/uhc_critical_care_policy.pdf"
  ]
}
```

## Expected Quality Targets

- **Story Alignment Quality**: 90%+ include business rule + fit explanation + situation + action
- **Citation Compliance**: 100% policy URLs (no business rules in citations)
- **Specificity Score**: 85%+ recommendations have concrete identifiers  
- **Brevity Compliance**: 100% adherence to word limits

## Output Files

- JSON export includes:
  - All recommendations
  - Quality metrics (including citation validation)
  - Attempt log (retry history)
  - Configuration metadata

## 5. Prompt Engineering Section

In [22]:
# ============= SYSTEM PROMPT (EDIT HERE) =============

SYSTEM_PROMPT = """You are an expert healthcare policy analyst specializing in healthcare policy recommendations across multiple domains including reimbursement policies, provider contracts, utilization management, and network optimization.

Your role is to:
- Analyze healthcare data and identify actionable policy opportunities
- Generate evidence-based recommendations grounded in the provided data
- Synthesize peer practices and industry benchmarks from the data
- Prioritize recommendations by financial impact and scope

Critical rules:
- Use ONLY information from the input data provided
- Do not hallucinate or add external knowledge
- Extract evidence directly from the data
- Output valid JSON only, no markdown or explanatory text
- All recommendations must be grounded in the input data
"""

print("SYSTEM PROMPT:")
print("="*80)
print(SYSTEM_PROMPT)
print("="*80)
print(f"Length: {len(SYSTEM_PROMPT)} characters")

SYSTEM PROMPT:
You are an expert healthcare policy analyst specializing in healthcare policy recommendations across multiple domains including reimbursement policies, provider contracts, utilization management, and network optimization.

Your role is to:
- Analyze healthcare data and identify actionable policy opportunities
- Generate evidence-based recommendations grounded in the provided data
- Synthesize peer practices and industry benchmarks from the data
- Prioritize recommendations by financial impact and scope

Critical rules:
- Use ONLY information from the input data provided
- Do not hallucinate or add external knowledge
- Extract evidence directly from the data
- Output valid JSON only, no markdown or explanatory text
- All recommendations must be grounded in the input data

Length: 781 characters


In [23]:
# ============= USER PROMPT TEMPLATES (EDIT HERE) =============

# Standard template (without decision tree)
USER_PROMPT_TEMPLATE = """You are analyzing healthcare data to generate policy recommendations.

INPUT DATA (structure may vary):
{input_json}

INSTRUCTIONS:
1. Analyze the input data and identify:
   - Patterns, findings, or issues (regardless of how they're labeled in the JSON)
   - Supporting evidence from claims/data (look for dollar amounts, percentages, volumes, growth rates)
   - Healthcare policies, rules, or contractual terms (from any section of the input)
   - Peer practices, industry benchmarks, or comparative information (if present)

2. Generate actionable policy recommendations:
   - Consolidate related patterns into single recommendations when they support the same action
   - Determine priority (HIGH/MEDIUM/LOW) based on:
     * Financial impact (dollar amounts, percentage of spend)
     * Scope (number of states, providers, members affected)
     * Urgency (policy gaps, peer adoption trends)
   - Rank recommendations by priority and impact (most impactful first)
   - Extract evidence bullets directly from the input data
   - Synthesize peer benchmarking or industry practices from the input
   - Write clear, concise description containing ONLY the recommended action (do not include next steps or peer support in description)

3. Use ONLY information from the input data - no external knowledge or assumptions

OUTPUT SCHEMA (REQUIRED):
{{
  "recommendations": [
    {{
      "rank": <number starting from 1>,
      "priority": "HIGH|MEDIUM|LOW",
      "category": "Policy",
      "description": "<clear, concise action statement describing what should be done>",
      "evidence": ["<bullet point from input data>", "<another bullet>", ...],
      "story_alignment": ["<how the pattern/data supports this recommendation>", ...],
      "peer_benchmarking": ["<peer payer name>: <their practice from input data>", ...]
    }}
  ]
}}

IMPORTANT:
- Every field in evidence, story_alignment, and peer_benchmarking must come from the input data
- Do not invent or assume information not present in the input
- If peer benchmarking data is not in the input, use empty array []
- Category must always be "Policy"
- Rank recommendations by priority (HIGH first, then MEDIUM, then LOW)
- Description field should contain ONLY the recommended action, not next steps or peer support

Return ONLY valid JSON matching this schema. No markdown, no explanations.
"""

# Enhanced template (with decision tree)
ENHANCED_USER_PROMPT_TEMPLATE = """You are analyzing healthcare data to generate policy recommendations.

INPUT DATA (structure may vary):
{input_json}

DECISION TREE RULES (CoC AI Analyst Guidelines):
{decision_tree_rules}

INSTRUCTIONS:
1. Identify which service categories apply to the input data (IP MedSurg, OP Surg, Phys PCP, Retail RX, etc.)
2. Match input patterns to relevant decision tree rules based on "Why?" descriptions
3. Use "Cost of Care Suggestions" from matched rules as recommendation templates
4. Enhance recommendations with specific evidence from input data
5. If no rules match, generate custom recommendations based on input analysis
6. Prioritize rule-based recommendations over custom ones

OUTPUT SCHEMA (REQUIRED):
{{
  "recommendations": [
    {{
      "rank": <number starting from 1>,
      "priority": "HIGH|MEDIUM|LOW",
      "category": "Policy",
      "description": "<clear action statement>",
      "evidence": ["<sentence from input>", "<another sentence>"],
      "story_alignment": ["<explanation>"],
      "peer_benchmarking": ["<peer info if available>"]
    }}
  ]
}}

CRITICAL FORMATTING RULES:
- DO NOT add serial numbers (1., 2., etc.) to list items
- DO NOT add bullet points (-, *, •) to list items
- Each list item should be a clean sentence without prefixes
- Example CORRECT: ["High paid volume identified", "Contract changes noted"]
- Example WRONG: ["1. High paid volume identified", "- Contract changes noted"]

Return ONLY valid JSON matching this schema.
"""

print("Available prompt templates:")
print(f"  1. USER_PROMPT_TEMPLATE (standard): {len(USER_PROMPT_TEMPLATE)} chars")
print(f"  2. ENHANCED_USER_PROMPT_TEMPLATE (with decision tree): {len(ENHANCED_USER_PROMPT_TEMPLATE)} chars")

Available prompt templates:
  1. USER_PROMPT_TEMPLATE (standard): 2357 chars
  2. ENHANCED_USER_PROMPT_TEMPLATE (with decision tree): 1477 chars


In [25]:
# Build final user prompt
if rule_engine and USE_DECISION_TREE:
    user_prompt = ENHANCED_USER_PROMPT_TEMPLATE.format(
        input_json=input_json_str,
        decision_tree_rules=rules_text
    )
    prompt_type = "ENHANCED (with decision tree)"
else:
    user_prompt = USER_PROMPT_TEMPLATE.format(input_json=input_json_str)
    prompt_type = "STANDARD (no decision tree)"

print(f"\n✓ User prompt built: {prompt_type}")
print(f"  Length: {len(user_prompt)} characters")
print(f"  Estimated tokens: ~{len(user_prompt) // 4}")

if SHOW_FULL_PROMPTS:
    print("\n" + "="*80)
    print(f"USER PROMPT ({prompt_type}):")
    print("="*80)
    print(user_prompt)


✓ User prompt built: ENHANCED (with decision tree)
  Length: 110023 characters
  Estimated tokens: ~27505

USER PROMPT (ENHANCED (with decision tree)):
You are analyzing healthcare data to generate policy recommendations.

INPUT DATA (structure may vary):
[
  {
    "rank": 1,
    "pattern_title": "High-Volume Critical Care Billing",
    "pattern_description": "High paid volume for CPT 99291 (Critical Care) with home discharge, suggesting potential inappropriate billing.",
    "service_category": "Phys Emergency (IP, OP)",
    "explanation": {
      "claim_evidence": {
        "summary": "Total exposure approximately $9.3M FI and $25.8M ASO across multiple states.",
        "details": [
          "High volume of 99291 claims with home discharge status",
          "Pattern observed across TX, FL, and CA markets",
          "Significant cost impact: $35.1M total exposure"
        ]
      },
      "reimbursement": {
        "individual_policies": [
          {
            "payer_name": "M

In [26]:
# Complete messages array for LLM
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_prompt}
]

print("COMPLETE MESSAGES ARRAY:")
print("="*80)
print(f"Message 1 (system): {len(messages[0]['content'])} chars")
print(f"Message 2 (user): {len(messages[1]['content'])} chars")
print(f"Total prompt size: {sum(len(m['content']) for m in messages)} chars")
print(f"Estimated total tokens: ~{sum(len(m['content']) for m in messages) // 4}")

if DEBUG_MODE:
    print("\nMessages structure:")
    print(json.dumps([{"role": m["role"], "content_length": len(m["content"])} for m in messages], indent=2))

COMPLETE MESSAGES ARRAY:
Message 1 (system): 781 chars
Message 2 (user): 110023 chars
Total prompt size: 110804 chars
Estimated total tokens: ~27701

Messages structure:
[
  {
    "role": "system",
    "content_length": 781
  },
  {
    "role": "user",
    "content_length": 110023
  }
]


## 6. LLM Invocation Section

In [28]:
from deep_research_agents.decision_tree_rules import DecisionTreeRuleEngine
from langchain_openai import ChatOpenAI
from deep_research_utils import EHAPBase
from deep_research_utils.app_constant import AppConstants
 
print("✓ Agent modules imported")
# LLM settings
LLM_REASONING_EFFORT = "high"  # "low", "medium", "high"
LLM_SUMMARY_MODE = None  # "auto", "detailed", or None

# Initialize EHAP authentication
print("Initializing EHAP authentication...")

EHAP = EHAPBase(
    base_url=AppConstants.EHAP_BASE_URL,
    client_id=AppConstants.EHAP_CLIENT_ID,
    client_secret=AppConstants.EHAP_CLIENT_SECRET,
    verify=AppConstants.SSL_CERT_FILE or False
)

print("✓ EHAP initialized")

# Initialize LLM
print(f"Initializing LLM with reasoning effort: {LLM_REASONING_EFFORT}")

llm = ChatOpenAI(
    base_url=AppConstants.OPENAI_BASE_URL,
    model=AppConstants.EHAP_LLM_MODEL,
    api_key=EHAP.get_token(),
    extra_body={
        "reasoning_effort": LLM_REASONING_EFFORT,
        "summary": LLM_SUMMARY_MODE
    },
    http_client=AppConstants.http_client_,
    http_async_client=AppConstants.http_async_client_,
)

print(f"✓ LLM initialized")
print(f"  Model: {AppConstants.EHAP_LLM_MODEL}")
print(f"  Reasoning effort: {LLM_REASONING_EFFORT}")

✓ Agent modules imported
Initializing EHAP authentication...
✓ EHAP initialized
Initializing LLM with reasoning effort: high
2026-05-11 16:10:12,381 - deep_research_utils.ehap - INFO - Requesting new access token from https://api.horizon.elevancehealth.com/v2/oauth2/token with client_id: iYfcOHQHPTaAn3BckTey49nJg93QsBmE


c:\Work_data\coc\dr_env_3_14\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.horizon.elevancehealth.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


2026-05-11 16:10:13,572 - deep_research_utils.ehap - INFO - Access token generated successfully.
**TOKEN** **TOKEN** **TOKEN** 
✓ LLM initialized
  Model: gpt-5.4
  Reasoning effort: high


In [29]:
# Invoke LLM
print("Sending request to LLM...")
start_time = datetime.now()

response = llm.invoke(messages)

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print(f"\n✓ LLM response received")
print(f"  Duration: {duration:.2f} seconds")
print(f"  Response length: {len(response.content)} characters")

if hasattr(response, 'response_metadata'):
    print(f"  Metadata: {response.response_metadata}")

Sending request to LLM...

✓ LLM response received
  Duration: 72.39 seconds
  Response length: 3618 characters
  Metadata: {'token_usage': {'completion_tokens': 4801, 'prompt_tokens': 21568, 'total_tokens': 26369, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 4013, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DeILx6yUXWFWTg8GLqxHypCGdJ77Z', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}


In [30]:
# Display raw LLM output
raw_output = response.content

print("="*80)
print("RAW LLM OUTPUT:")
print("="*80)
print(raw_output)
print("="*80)
print(f"Length: {len(raw_output)} characters")
print(f"Lines: {len(raw_output.split(chr(10)))}")

RAW LLM OUTPUT:
{
  "recommendations": [
    {
      "rank": 1,
      "priority": "HIGH",
      "category": "Policy",
      "description": "Implement an emergency critical care reimbursement policy and claims edit for CPT 99291 claims with home discharge, denying or routing claims to medical review when admission or transfer criteria are not met.",
      "evidence": [
        "High volume of 99291 claims with home discharge status",
        "Pattern observed across TX, FL, and CA markets",
        "Significant cost impact: $35.1M total exposure"
      ],
      "story_alignment": [
        "Mapped Phys Emergency (IP, OP) to the OP ER decision tree because the issue centers on emergency critical care billing.",
        "The pattern reflects high paid volume of a high-severity emergency service with home discharge, which aligns to the OP ER rule on increased utility of higher severity codes and supports an ER leveling or reimbursement edit.",
        "Peer reimbursement evidence in the in

## 7. Response Processing Section

In [31]:
# Clean response (remove markdown fences)
cleaned = raw_output.strip()

print("Cleaning response...")

if cleaned.startswith("```"):
    print("  Found markdown code fences, removing...")
    lines = cleaned.split("\n")
    if len(lines) > 2:
        if lines[0].startswith("```json"):
            cleaned = "\n".join(lines[1:-1])
        else:
            cleaned = "\n".join(lines[1:-1])
    print("  ✓ Markdown fences removed")
else:
    print("  No markdown fences found")

print(f"\nCleaned output length: {len(cleaned)} characters")

if SHOW_FULL_PROMPTS:
    print("\n" + "="*80)
    print("CLEANED OUTPUT (first 500 chars):")
    print("="*80)
    print(cleaned[:500] + "...")

Cleaning response...
  No markdown fences found

Cleaned output length: 3618 characters

CLEANED OUTPUT (first 500 chars):
{
  "recommendations": [
    {
      "rank": 1,
      "priority": "HIGH",
      "category": "Policy",
      "description": "Implement an emergency critical care reimbursement policy and claims edit for CPT 99291 claims with home discharge, denying or routing claims to medical review when admission or transfer criteria are not met.",
      "evidence": [
        "High volume of 99291 claims with home discharge status",
        "Pattern observed across TX, FL, and CA markets",
        "Significant ...


In [32]:
# Parse JSON
print("Parsing JSON...")

try:
    result = json.loads(cleaned)
    print("✓ JSON parsed successfully")
    
    if not isinstance(result, dict):
        print("⚠ Warning: Result is not a dictionary")
        result = {"recommendations": []}
    
    if "recommendations" not in result:
        print("⚠ Warning: Missing 'recommendations' key")
        result = {"recommendations": []}
    else:
        rec_count = len(result["recommendations"])
        print(f"  Found {rec_count} recommendation(s)")
        
except json.JSONDecodeError as e:
    print(f"❌ JSON parsing failed: {e}")
    print(f"   Error at line {e.lineno}, column {e.colno}")
    result = {"recommendations": []}

Parsing JSON...
✓ JSON parsed successfully
  Found 3 recommendation(s)


In [33]:
# Clean list formatting function
def clean_list_formatting(text: str) -> str:
    """Remove serial numbers, bullets, and formatting from text."""
    if not text or not isinstance(text, str):
        return text
    
    # Remove leading numbers with dots/parens: "1. ", "1) ", "(1) "
    text = re.sub(r'^\s*\(?\d+[\.\)\)]\s*', '', text.strip())
    # Remove bullet points: "- ", "* ", "• "
    text = re.sub(r'^\s*[-*•]\s+', '', text.strip())
    # Remove markdown list markers
    text = re.sub(r'^\s*[\-\*\+]\s+', '', text.strip())
    
    return text.strip()

# Apply cleaning to all list fields
print("Cleaning list formatting...")
cleaned_count = 0

for rec in result.get('recommendations', []):
    if not isinstance(rec, dict):
        continue
    
    for field in ['evidence', 'story_alignment', 'peer_benchmarking']:
        if field in rec and isinstance(rec[field], list):
            before = rec[field].copy()
            rec[field] = [clean_list_formatting(item) for item in rec[field] if item]
            
            # Check if any cleaning occurred
            if before != rec[field]:
                cleaned_count += 1

if cleaned_count > 0:
    print(f"  ✓ Cleaned {cleaned_count} list field(s)")
else:
    print("  No formatting issues found")

print("\n✓ All list formatting cleaned")

Cleaning list formatting...
  No formatting issues found

✓ All list formatting cleaned


In [34]:
# Validate output schema
def validate_output_schema(result: Dict[str, Any]) -> List[str]:
    """Validate output matches expected schema."""
    errors = []
    
    if not isinstance(result, dict):
        errors.append("Result is not a dictionary")
        return errors
    
    if "recommendations" not in result:
        errors.append("Missing 'recommendations' key")
        return errors
    
    recommendations = result["recommendations"]
    
    if not isinstance(recommendations, list):
        errors.append("'recommendations' is not a list")
        return errors
    
    required_fields = ["rank", "priority", "category", "description", "evidence", "story_alignment", "peer_benchmarking"]
    
    for i, rec in enumerate(recommendations):
        if not isinstance(rec, dict):
            errors.append(f"Recommendation {i} is not a dictionary")
            continue
        
        for field in required_fields:
            if field not in rec:
                errors.append(f"Recommendation {i} missing required field: {field}")
        
        if "priority" in rec and rec["priority"] not in ["HIGH", "MEDIUM", "LOW"]:
            errors.append(f"Recommendation {i} has invalid priority: {rec['priority']}")
        
        if "category" in rec and rec["category"] != "Policy":
            errors.append(f"Recommendation {i} has non-Policy category: {rec['category']}")
        
        for list_field in ["evidence", "story_alignment", "peer_benchmarking"]:
            if list_field in rec and not isinstance(rec[list_field], list):
                errors.append(f"Recommendation {i} field '{list_field}' is not a list")
    
    return errors

# Perform validation
print("Validating output schema...")
validation_errors = validate_output_schema(result)

if validation_errors:
    print(f"\n⚠ Found {len(validation_errors)} validation issue(s):")
    for error in validation_errors:
        print(f"  - {error}")
else:
    print("\n✓ Output schema validation passed")

Validating output schema...

✓ Output schema validation passed


## 8. Results Display Section

In [35]:
# Display final recommendations (JSON format)
print("="*80)
print("FINAL RECOMMENDATIONS (JSON):")
print("="*80)
print(json.dumps(result, indent=2))
print("="*80)

# Priority breakdown
if result.get('recommendations'):
    priorities = {}
    for rec in result['recommendations']:
        priority = rec.get('priority', 'UNKNOWN')
        priorities[priority] = priorities.get(priority, 0) + 1
    
    print(f"\nPriority Breakdown:")
    for priority, count in sorted(priorities.items()):
        print(f"  {priority}: {count}")

FINAL RECOMMENDATIONS (JSON):
{
  "recommendations": [
    {
      "rank": 1,
      "priority": "HIGH",
      "category": "Policy",
      "description": "Implement an emergency critical care reimbursement policy and claims edit for CPT 99291 claims with home discharge, denying or routing claims to medical review when admission or transfer criteria are not met.",
      "evidence": [
        "High volume of 99291 claims with home discharge status",
        "Pattern observed across TX, FL, and CA markets",
        "Significant cost impact: $35.1M total exposure"
      ],
      "story_alignment": [
        "Mapped Phys Emergency (IP, OP) to the OP ER decision tree because the issue centers on emergency critical care billing.",
        "The pattern reflects high paid volume of a high-severity emergency service with home discharge, which aligns to the OP ER rule on increased utility of higher severity codes and supports an ER leveling or reimbursement edit.",
        "Peer reimbursement evid

In [36]:
# Detailed recommendation view (formatted)
print("="*80)
print("DETAILED RECOMMENDATIONS VIEW:")
print("="*80)

for rec in result.get('recommendations', []):
    print(f"\n{'='*80}")
    print(f"Rank {rec.get('rank', '?')} [{rec.get('priority', 'UNKNOWN')}] - {rec.get('category', 'N/A')}")
    print(f"{'='*80}")
    
    print(f"\nDescription:")
    print(f"  {rec.get('description', 'N/A')}")
    
    print(f"\nEvidence:")
    for item in rec.get('evidence', []):
        print(f"  • {item}")
    
    print(f"\nStory Alignment:")
    for item in rec.get('story_alignment', []):
        print(f"  • {item}")
    
    print(f"\nPeer Benchmarking:")
    peer_items = rec.get('peer_benchmarking', [])
    if peer_items:
        for item in peer_items:
            print(f"  • {item}")
    else:
        print(f"  (none)")

print(f"\n{'='*80}")

DETAILED RECOMMENDATIONS VIEW:

Rank 1 [HIGH] - Policy

Description:
  Implement an emergency critical care reimbursement policy and claims edit for CPT 99291 claims with home discharge, denying or routing claims to medical review when admission or transfer criteria are not met.

Evidence:
  • High volume of 99291 claims with home discharge status
  • Pattern observed across TX, FL, and CA markets
  • Significant cost impact: $35.1M total exposure

Story Alignment:
  • Mapped Phys Emergency (IP, OP) to the OP ER decision tree because the issue centers on emergency critical care billing.
  • The pattern reflects high paid volume of a high-severity emergency service with home discharge, which aligns to the OP ER rule on increased utility of higher severity codes and supports an ER leveling or reimbursement edit.
  • Peer reimbursement evidence in the input supports excluding or challenging critical care when the patient is discharged home.

Peer Benchmarking:
  • Deny critical care in ED

## 9. Experimentation Section

In [37]:
# Compare with/without decision tree
print("COMPARISON: With vs Without Decision Tree")
print("="*80)
print("\nTo compare:")
print("1. Run this notebook once with USE_DECISION_TREE = True")
print("2. Note the recommendations")
print("3. Change USE_DECISION_TREE = False in Cell 2")
print("4. Re-run from Cell 5 onwards")
print("5. Compare the results")
print("\nKey differences to look for:")
print("  - Recommendation descriptions (rule-based vs custom)")
print("  - Priority assignments")
print("  - Evidence extraction")
print("  - Number of recommendations generated")

COMPARISON: With vs Without Decision Tree

To compare:
1. Run this notebook once with USE_DECISION_TREE = True
2. Note the recommendations
3. Change USE_DECISION_TREE = False in Cell 2
4. Re-run from Cell 5 onwards
5. Compare the results

Key differences to look for:
  - Recommendation descriptions (rule-based vs custom)
  - Priority assignments
  - Evidence extraction
  - Number of recommendations generated


In [38]:
# Prompt tweaking playground
print("PROMPT TWEAKING PLAYGROUND")
print("="*80)
print("\nTo experiment with prompts:")
print("\n1. Edit SYSTEM_PROMPT in Cell 8:")
print("   - Try different role descriptions")
print("   - Add/remove constraints")
print("   - Change output requirements")
print("\n2. Edit USER_PROMPT_TEMPLATE or ENHANCED_USER_PROMPT_TEMPLATE in Cell 9:")
print("   - Modify instruction structure")
print("   - Change priority criteria")
print("   - Adjust output schema")
print("   - Add/remove formatting rules")
print("\n3. Re-run from Cell 10 onwards to see impact")
print("\n4. Compare outputs and iterate")
print("\nTips:")
print("  - Make small changes and test")
print("  - Keep notes of what works/doesn't work")
print("  - Check validation errors after each change")
print("  - Monitor prompt size (stay under model limits)")

PROMPT TWEAKING PLAYGROUND

To experiment with prompts:

1. Edit SYSTEM_PROMPT in Cell 8:
   - Try different role descriptions
   - Add/remove constraints
   - Change output requirements

2. Edit USER_PROMPT_TEMPLATE or ENHANCED_USER_PROMPT_TEMPLATE in Cell 9:
   - Modify instruction structure
   - Change priority criteria
   - Adjust output schema
   - Add/remove formatting rules

3. Re-run from Cell 10 onwards to see impact

4. Compare outputs and iterate

Tips:
  - Make small changes and test
  - Keep notes of what works/doesn't work
  - Check validation errors after each change
  - Monitor prompt size (stay under model limits)


In [39]:
# Input variation testing
print("INPUT VARIATION TESTING")
print("="*80)
print("\nTo test different input structures:")
print("\n1. Edit input_data in Cell 3:")
print("   - Add/remove patterns")
print("   - Change service categories")
print("   - Vary evidence format")
print("   - Test with/without peer benchmarking")
print("\n2. Re-run from Cell 4 onwards")
print("\n3. Observe:")
print("   - How many recommendations are generated")
print("   - Which decision tree rules match (if enabled)")
print("   - Quality of evidence extraction")
print("   - Priority assignments")
print("\nTest cases to try:")
print("  - Single pattern vs multiple patterns")
print("  - Different service categories")
print("  - Varying financial impact levels")
print("  - With/without peer benchmarking data")

INPUT VARIATION TESTING

To test different input structures:

1. Edit input_data in Cell 3:
   - Add/remove patterns
   - Change service categories
   - Vary evidence format
   - Test with/without peer benchmarking

2. Re-run from Cell 4 onwards

3. Observe:
   - How many recommendations are generated
   - Which decision tree rules match (if enabled)
   - Quality of evidence extraction
   - Priority assignments

Test cases to try:
  - Single pattern vs multiple patterns
  - Different service categories
  - Varying financial impact levels
  - With/without peer benchmarking data


## Summary

### What You Can Experiment With:

1. **Prompts** (Cells 8-9)
   - System prompt instructions
   - User prompt templates
   - Output schema requirements

2. **Decision Tree** (Cells 5-7)
   - Enable/disable rules
   - View rule structure
   - See formatted rules text

3. **Input Data** (Cell 3)
   - Different patterns
   - Various formats
   - Different service categories

4. **LLM Settings** (Cell 2)
   - Model selection
   - Temperature
   - Reasoning effort

### Visibility at Each Step:

✅ Input validation and serialization  
✅ Decision tree rules loading and formatting  
✅ Complete prompt construction  
✅ Raw LLM output  
✅ Response cleaning and parsing  
✅ List formatting cleanup  
✅ Schema validation  
✅ Final recommendations  

### Next Steps:

- Tweak prompts and compare outputs
- Test different input structures
- A/B test with/without decision tree
- Refine based on results
- Apply learnings back to the agent code